# Compare SDE PSD Models

Compare notebook-trained PSD models in both pure-noise DDPM sampling and restoration-style DDPM sampling using the original `Diffuser.py`.


In [ ]:
from pathlib import Path
import sys
import re
import math

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

try:
    import pandas as pd
except Exception:
    pd = None


def find_repo_root() -> Path:
    candidates = [
        Path.cwd().resolve(),
        Path.cwd().resolve() / "SpecDiff",
        Path("/Users/27171653/Desktop/PhD/Highlight-modelling/Specular-Highlights/SpecDiff"),
    ]
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "Run_Training.py").exists() and (candidate / "SDEBackbone.py").exists() and (candidate / "Diffuser.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the SpecDiff repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import SDEBackbone
import Diffuser as diff

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEFAULT_PSD_ROOT = Path("/Users/27171653/Desktop/PhD/Highlight-modelling/PSD_Dataset/PSD_Dataset")
PSD_ROOT = DEFAULT_PSD_ROOT
MODEL_DIRS = [ROOT / "models" / "32", ROOT / "checkpoints"]
FIXED_CASE_INDEX = 0
MAX_CASES = None
PREFER_EMA = True
SHOW_PROGRESS = True
PURE_NOISE_BASE_SEED = 123
RESTORE_BASE_SEED = 9123
RESTORE_START_STEP = 40

print(f"repo root: {ROOT}")
print(f"device: {DEVICE}")
print(f"PSD root: {PSD_ROOT}")


In [ ]:
VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
PIL_BILINEAR = getattr(Image, "Resampling", Image).BILINEAR


def normalize_image_key(name: str) -> str:
    stem = Path(name).stem.lower()
    stem = stem.replace("specular", "").replace("glossy", "").replace("diffuse", "")
    return re.sub(r"[^a-z0-9]+", "", stem)


def pair_image_paths(glossy_dir: Path, diffuse_dir: Path):
    glossy_candidates = [path for path in glossy_dir.iterdir() if path.suffix.lower() in VALID_EXTS]
    diffuse_candidates = [path for path in diffuse_dir.iterdir() if path.suffix.lower() in VALID_EXTS]

    glossy_files = {path.name: path for path in glossy_candidates}
    diffuse_files = {path.name: path for path in diffuse_candidates}
    exact_names = sorted(set(glossy_files) & set(diffuse_files))
    if exact_names:
        return [(glossy_files[name], diffuse_files[name], name) for name in exact_names]

    glossy_by_key = {}
    for path in glossy_candidates:
        key = normalize_image_key(path.name)
        if key in glossy_by_key:
            raise RuntimeError(f"Duplicate glossy normalized key {key!r} in {glossy_dir}")
        glossy_by_key[key] = path

    diffuse_by_key = {}
    for path in diffuse_candidates:
        key = normalize_image_key(path.name)
        if key in diffuse_by_key:
            raise RuntimeError(f"Duplicate diffuse normalized key {key!r} in {diffuse_dir}")
        diffuse_by_key[key] = path

    common_keys = sorted(set(glossy_by_key) & set(diffuse_by_key))
    if not common_keys:
        raise RuntimeError(f"No paired PSD samples found in {glossy_dir} and {diffuse_dir}")
    return [(glossy_by_key[key], diffuse_by_key[key], glossy_by_key[key].name) for key in common_keys]


def load_rgb_tensor(path: Path, image_size: int) -> torch.Tensor:
    image = Image.open(path).convert("RGB")
    image = image.resize((image_size, image_size), resample=PIL_BILINEAR)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1).contiguous()


def build_condition_and_target(glossy: torch.Tensor, diffuse: torch.Tensor, task_mode: str):
    if task_mode == "direct_diffuse":
        return glossy, diffuse, diffuse
    if task_mode == "difference":
        return glossy, glossy - diffuse, diffuse
    if task_mode == "diffuse_prior":
        return torch.zeros_like(diffuse), diffuse, diffuse
    raise ValueError(f"Unknown task mode: {task_mode}")


def build_restore_prior(glossy: torch.Tensor, condition: torch.Tensor, task_mode: str) -> torch.Tensor:
    if task_mode == "difference":
        return torch.zeros_like(condition)
    if task_mode == "diffuse_prior":
        return glossy.clone()
    return condition.clone()


def prediction_to_diffuse(prediction: torch.Tensor, condition: torch.Tensor, task_mode: str) -> torch.Tensor:
    if task_mode == "difference":
        return (condition - prediction).clamp(0.0, 1.0)
    return prediction.clamp(0.0, 1.0)


def discover_model_files(model_dirs):
    discovered = []
    for directory in model_dirs:
        if not directory.exists():
            continue
        for path in sorted(directory.rglob("*.pth")):
            try:
                checkpoint = torch.load(path, map_location="cpu")
            except Exception:
                continue
            if not isinstance(checkpoint, dict):
                continue
            if checkpoint.get("notebook_group") != "sde_psd_normal_diffuser":
                continue
            discovered.append((path, checkpoint))
    return discovered


def extract_state_dict(checkpoint: dict):
    if PREFER_EMA and "ema_model_state_dict" in checkpoint:
        return checkpoint["ema_model_state_dict"], "ema_model_state_dict"
    return checkpoint["model_state_dict"], "model_state_dict"


def model_label(path: Path) -> str:
    return path.name if path.name != "model.pth" else path.parent.name


def make_case_noise(shape, seed: int, dtype: torch.dtype = torch.float32) -> torch.Tensor:
    gen = torch.Generator()
    gen.manual_seed(int(seed))
    return torch.randn(shape, generator=gen, dtype=dtype)


def ddpm_sample_safe(model: torch.nn.Module, diffuser: diff.Diffuser, condition: torch.Tensor, initial_noise: torch.Tensor | None = None) -> torch.Tensor:
    with torch.no_grad():
        if initial_noise is None:
            x_t = torch.randn_like(condition)
        else:
            x_t = initial_noise.to(condition.device, dtype=condition.dtype).clone()
        t_now = torch.full((x_t.shape[0],), diffuser.steps - 1, device=condition.device, dtype=torch.long)
        for _ in range(diffuser.steps - 1):
            t_pre = torch.clamp(t_now - 1, min=0)
            predicted_noise = model(x_t, t_now, condition)
            x_t = diffuser.DDPM_sample_step(x_t, t_now, t_pre, predicted_noise)
            t_now = t_pre
        return x_t


def ddpm_restore_safe(model: torch.nn.Module, diffuser: diff.Diffuser, condition: torch.Tensor, glossy: torch.Tensor, task_mode: str, start_step: int, seed: int) -> torch.Tensor:
    with torch.no_grad():
        start_step = int(max(1, min(start_step, diffuser.steps - 1)))
        x0_prior = build_restore_prior(glossy, condition, task_mode)
        t_now = torch.full((condition.shape[0],), start_step, device=condition.device, dtype=torch.long)
        noise = make_case_noise(tuple(x0_prior.shape), seed=seed, dtype=x0_prior.dtype).to(condition.device)
        x_t = diffuser.forward_diffusion(x0_prior, t_now, noise)
        for _ in range(start_step):
            t_pre = torch.clamp(t_now - 1, min=0)
            predicted_noise = model(x_t, t_now, condition)
            x_t = diffuser.DDPM_sample_step(x_t, t_now, t_pre, predicted_noise)
            t_now = t_pre
        return x_t


def compute_metrics(prediction: torch.Tensor, target: torch.Tensor, pred_diffuse: torch.Tensor, true_diffuse: torch.Tensor) -> dict:
    target_mse = F.mse_loss(prediction, target).item()
    diffuse_mse = F.mse_loss(pred_diffuse, true_diffuse).item()
    return {
        "target_mse": float(target_mse),
        "target_mae": float((prediction - target).abs().mean().item()),
        "diffuse_mse": float(diffuse_mse),
        "diffuse_mae": float((pred_diffuse - true_diffuse).abs().mean().item()),
        "diffuse_psnr": float(10.0 * math.log10(1.0 / max(diffuse_mse, 1e-10))),
    }


glossy_dir = PSD_ROOT / "PSD_Test" / "PSD_Test_specular"
diffuse_dir = PSD_ROOT / "PSD_Test" / "PSD_Test_diffuse"
pairs = pair_image_paths(glossy_dir, diffuse_dir)
if MAX_CASES is not None:
    pairs = pairs[:int(MAX_CASES)]
print(f"test pairs: {len(pairs)}")

found = discover_model_files(MODEL_DIRS)
print(f"model files found: {len(found)}")


In [ ]:
results = []
visuals = []

for path, checkpoint in tqdm(found, disable=not SHOW_PROGRESS, desc="Models", dynamic_ncols=True):
    task_mode = checkpoint["task_mode"]
    image_size = int(checkpoint.get("image_size", 32))
    noise_steps = int(checkpoint.get("noise_steps", 200))
    depth = int(checkpoint.get("depth", 4))
    state_dict, state_key = extract_state_dict(checkpoint)

    model = SDEBackbone.UNetWithTransformer(noise_steps=noise_steps, size=image_size, depth=depth, conditioning_channels=3).to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    diffuser = diff.CosSchDiffuser(steps=noise_steps, device=DEVICE)

    pure_rows = []
    restore_rows = []
    for idx, (glossy_path, diffuse_path, case_name) in enumerate(pairs):
        glossy = load_rgb_tensor(glossy_path, image_size)
        diffuse_img = load_rgb_tensor(diffuse_path, image_size)
        condition, target, true_diffuse = build_condition_and_target(glossy, diffuse_img, task_mode)

        condition = condition.unsqueeze(0).to(DEVICE)
        target = target.unsqueeze(0).to(DEVICE)
        true_diffuse = true_diffuse.unsqueeze(0).to(DEVICE)

        pure_seed = PURE_NOISE_BASE_SEED + idx
        restore_seed = RESTORE_BASE_SEED + idx
        pure_noise = make_case_noise(tuple(target.shape), seed=pure_seed, dtype=target.dtype).to(DEVICE)

        with torch.no_grad():
            pure_prediction = ddpm_sample_safe(model, diffuser, condition, initial_noise=pure_noise)
            restore_prediction = ddpm_restore_safe(model, diffuser, condition, glossy.unsqueeze(0).to(DEVICE), task_mode, start_step=RESTORE_START_STEP, seed=restore_seed)

        pure_pred_diffuse = prediction_to_diffuse(pure_prediction, condition, task_mode)
        restore_pred_diffuse = prediction_to_diffuse(restore_prediction, condition, task_mode)

        pure_rows.append(compute_metrics(pure_prediction, target, pure_pred_diffuse, true_diffuse))
        restore_rows.append(compute_metrics(restore_prediction, target, restore_pred_diffuse, true_diffuse))

        if idx == FIXED_CASE_INDEX:
            visuals.append({
                "name": model_label(path),
                "task_mode": task_mode,
                "condition": condition.squeeze(0).cpu(),
                "true_diffuse": true_diffuse.squeeze(0).cpu(),
                "pure_pred_diffuse": pure_pred_diffuse.squeeze(0).cpu(),
                "restore_pred_diffuse": restore_pred_diffuse.squeeze(0).cpu(),
            })

    summary = {
        "model": model_label(path),
        "task_mode": task_mode,
        "state_key": state_key,
        "num_cases": len(pure_rows),
        "restore_start_step": RESTORE_START_STEP,
    }
    for prefix, rows in (("pure", pure_rows), ("restore", restore_rows)):
        for key in rows[0].keys():
            summary[f"{prefix}_{key}"] = float(np.mean([row[key] for row in rows]))
    results.append(summary)

if pd is not None and results:
    display(pd.DataFrame(results).sort_values(["restore_diffuse_psnr", "pure_diffuse_psnr"], ascending=False).reset_index(drop=True))
else:
    for row in sorted(results, key=lambda item: item["restore_diffuse_psnr"], reverse=True):
        print(row)


In [ ]:
if visuals:
    fig, axes = plt.subplots(len(visuals), 6, figsize=(20, max(4, 3.5 * len(visuals))), squeeze=False)
    titles = ["Condition", "True diffuse", "Pure-noise diffuse", "Pure-noise abs error", "Restore diffuse", "Restore abs error"]
    for col, title in enumerate(titles):
        axes[0, col].set_title(title)

    for row_idx, item in enumerate(visuals):
        condition = item["condition"].clamp(0.0, 1.0)
        true_diffuse = item["true_diffuse"].clamp(0.0, 1.0)
        pure_pred = item["pure_pred_diffuse"].clamp(0.0, 1.0)
        restore_pred = item["restore_pred_diffuse"].clamp(0.0, 1.0)
        pure_err = (pure_pred - true_diffuse).abs().mean(dim=0)
        restore_err = (restore_pred - true_diffuse).abs().mean(dim=0)

        axes[row_idx, 0].imshow(condition.permute(1, 2, 0).numpy())
        axes[row_idx, 1].imshow(true_diffuse.permute(1, 2, 0).numpy())
        axes[row_idx, 2].imshow(pure_pred.permute(1, 2, 0).numpy())
        axes[row_idx, 3].imshow(pure_err.numpy(), cmap="magma")
        axes[row_idx, 4].imshow(restore_pred.permute(1, 2, 0).numpy())
        axes[row_idx, 5].imshow(restore_err.numpy(), cmap="magma")
        axes[row_idx, 0].set_ylabel(f"{item['name']}\n{item['task_mode']}", rotation=0, labelpad=70, va="center")

    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No visuals to show.")
